# Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Estacion y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2: Imputación de valores NaN según duración del gap
y dos estrategias: por estación individual y global (todas las estaciones).

Requisitos:
- pandas, numpy, scikit-learn (para KNNImputer)
- Los archivos de entrada son los generados en el Código 1 (*_outliers.csv)
  que contienen la columna 'O3_for_impute' con los NaN de errores de sensor.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# PARÁMETROS CONFIGURABLES
# ============================================================================

# Carpeta donde están los archivos con outliers (salida del Código 1)
INPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")

# Carpetas de salida para los datos imputados
OUTPUT_BY_STATION = os.path.join(INPUT_DIR, "imputed_by_station")
OUTPUT_GLOBAL = os.path.join(INPUT_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_STATION, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales para los métodos de imputación (en horas)
SHORT_GAP_MAX = 6          # ≤ 6h → interpolación lineal
MEDIUM_GAP_MAX = 72        # >6h y <72h → estacional por hora
LONG_GAP_MIN = 72          # ≥72h → multivariante (KNN)

# Método para la imputación estacional: 'mean' o 'median'
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye O3_for_impute, que será la O3 final)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute',   # O3 se imputa desde esta columna
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Columnas que se conservan pero no se imputan (categóricas, flags, etc.)
# (Se mantienen intactas en el output)

# Para KNNImputer (multivariante)
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filename):
    """Carga un archivo CSV de una estación (con índice datetime)."""
    df = pd.read_csv(filename, index_col=0, parse_dates=True)
    # Asegurar que el índice es datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un diccionario con los grupos de NaN y su longitud.
    Cada grupo es un objeto Index con las posiciones del gap.
    """
    mask = series.isna()
    # Identificar grupos consecutivos de True (NaN)
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)   # guardamos el índice como tupla para poder iterar
    return gap_groups

def impute_by_station(df, use_global_hourly_stats=None):
    """
    Imputa un DataFrame de una sola estación aplicando los métodos en orden:
    1. Gaps cortos (≤6h) → interpolación lineal.
    2. Gaps medios (6-72h) → media/mediana por hora (estacional).
    3. Gaps largos (≥72h) → se dejan para KNN (se aplica después externamente).

    Si use_global_hourly_stats es un dict (hour -> valor), se usan esas medias globales;
    de lo contrario, se calculan a partir de los propios datos de la estación.
    """
    df_imp = df.copy()
    # Procesar cada columna numérica
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        # Obtener gaps
        gaps = get_gap_lengths(serie)
        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)  # convertir a Index
            if L <= SHORT_GAP_MAX:
                # Interpolación lineal
                # interpolate con limit_direction='both' rellena usando valores anteriores y posteriores
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                df_imp.loc[idx, col] = serie_interp.loc[idx]
            elif L < MEDIUM_GAP_MAX:
                # Imputación estacional por hora
                if use_global_hourly_stats is not None:
                    # Usar estadísticos globales
                    hourly_vals = idx.hour.map(use_global_hourly_stats)
                else:
                    # Calcular estadísticos por hora de la propia estación
                    # Excluimos los valores del gap actual (aunque son NaN, no influyen)
                    if SEASONAL_STAT == 'mean':
                        hourly_stats = serie.dropna().groupby(serie.dropna().index.hour).mean()
                    else:
                        hourly_stats = serie.dropna().groupby(serie.dropna().index.hour).median()
                    hourly_vals = idx.hour.map(hourly_stats)
                df_imp.loc[idx, col] = hourly_vals.values
            else:
                # Gaps largos: se dejarán para KNN (no hacemos nada aquí)
                pass
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas numéricas del DataFrame.
    extra_cols: columnas adicionales que se incluirán como predictores pero no se imputarán (ej. station_id).
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if extra_cols:
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    # Seleccionar subconjunto
    sub = df[all_cols].copy()
    # KNNImputer solo funciona con datos numéricos; ya lo son.
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    imputed_array = imputer.fit_transform(sub)
    # Reasignar solo las columnas imputadas (las extra no se modifican)
    df_imp = df.copy()
    df_imp[cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

# ============================================================================
# PROCESAMIENTO POR ESTACIÓN (INDIVIDUAL)
# ============================================================================

def process_by_station():
    print("\n=== Imputación por estación (individual) ===")
    # Buscar todos los archivos _outliers.csv en INPUT_DIR
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        print(f"Procesando {station_name} ...")
        df = load_station_data(filepath)
        # Imputar con estadísticos propios
        df_imp = impute_by_station(df, use_global_hourly_stats=None)
        # Aplicar KNN a los NaN restantes (gaps largos)
        # Nota: KNN se aplica a todas las filas, pero solo modificará NaN
        df_imp = apply_knn_imputation(df_imp, NUM_COLS)
        # Renombrar O3_for_impute a O3 para los modelos
        if 'O3_for_impute' in df_imp.columns:
            df_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)
        # Guardar
        out_path = os.path.join(OUTPUT_BY_STATION, f"{station_name}.csv")
        df_imp.to_csv(out_path, index=True)
        print(f"  Guardado en {out_path}")

# ============================================================================
# PROCESAMIENTO GLOBAL (TODAS LAS ESTACIONES JUNTAS)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Lista para concatenar
    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)
        df['station_id'] = station_name  # identificador de estación
        dfs.append(df)

    # Concatenar todos
    df_all = pd.concat(dfs, axis=0)
    # Ordenar por datetime y station_id? No es necesario, pero para KNN puede ayudar
    # Dejamos como está.

    # 1. Calcular estadísticos horarios globales (media/mediana por hora) para cada variable
    #    usando todos los datos no NaN de todas las estaciones.
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        if SEASONAL_STAT == 'mean':
            hourly = s.groupby(s.index.hour).mean()
        else:
            hourly = s.groupby(s.index.hour).median()
        global_hourly_stats[col] = hourly

    # 2. Imputar cada estación por separado usando los estadísticos globales
    #    (para interpolación y estacional). Hacemos loop sobre las estaciones.
    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        # Aplicar imputación con estadísticos globales (pasamos el dict de medias por hora)
        # Necesitamos una versión de impute_by_station que acepte un dict con las medias por columna.
        # Modificamos la función para que acepte un dict de series indexadas por hora.
        # Para simplificar, reimplementamos aquí un bucle específico.
        for col in NUM_COLS:
            if col not in df_station.columns:
                continue
            serie = df_station[col]
            if serie.isna().sum() == 0:
                continue
            gaps = get_gap_lengths(serie)
            for idx_tuple, L in gaps.items():
                idx = pd.Index(idx_tuple)
                if L <= SHORT_GAP_MAX:
                    # Interpolación lineal dentro de la estación (usamos la serie completa de la estación)
                    serie_interp = serie.interpolate(method='linear', limit_direction='both')
                    df_station.loc[idx, col] = serie_interp.loc[idx]
                elif L < MEDIUM_GAP_MAX:
                    # Usar estadístico global por hora
                    hourly_vals = idx.hour.map(global_hourly_stats[col])
                    df_station.loc[idx, col] = hourly_vals.values
                # largos se dejan para KNN
        # Actualizar el DataFrame global
        df_all_imp.loc[mask] = df_station

    # 3. Aplicar KNN imputer a todo el conjunto (incluyendo station_id como predictor)
    #    para los gaps largos restantes.
    #    Incluimos station_id como variable numérica (convertimos a código)
    #    Primero, convertir station_id a código numérico (0,1,2,...)
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    # Aplicar KNN
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    # Eliminar station_code (ya no necesario)
    df_all_imp.drop(columns=['station_code'], inplace=True)

    # 4. Separar por estación y guardar
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        df_station.drop(columns=['station_id'], inplace=True)
        # Renombrar O3_for_impute a O3
        if 'O3_for_impute' in df_station.columns:
            df_station.rename(columns={'O3_for_impute': 'O3'}, inplace=True)
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado {station} en {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos...")
    process_by_station()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_STATION}")
    print(f"  - {OUTPUT_GLOBAL}")

# Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2 (adaptado a transectos): Imputación de valores NaN según duración del gap.
Ahora se generan dos tipos de datasets:
  - Por transecto: un único archivo por transecto con todas sus estaciones.
  - Global: un archivo por estación (como antes) pero usando datos de todas las estaciones.

Entrada: Archivos *_outliers.csv (de Código 1) en Datos_TFG_outliers/
Salida:
  - imputed_by_transect/ : archivos Transecto_1.csv, Transecto_2.csv, ... (con columna Estacion)
  - imputed_global/      : archivos por estación (igual que antes)
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Volumes/copia seguridad1/TFG_Prueba/Datos_TFG_outliers")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye O3_for_impute)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES (comunes)
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un diccionario {índice_tupla: longitud} para cada grupo consecutivo de NaN.
    """
    mask = series.isna()
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def impute_dataframe(df, hourly_stats_dict):
    """
    Aplica imputación a un DataFrame:
      - Gaps cortos → interpolación lineal (usando la propia serie).
      - Gaps medios → relleno con estadístico horario (desde hourly_stats_dict).
      - Gaps largos → se dejan para KNN (no se modifican aquí).
    hourly_stats_dict: dict {nombre_columna: Series indexada por hora (0-23)}
    """
    df_imp = df.copy()
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)
            if L <= SHORT_GAP_MAX:
                # Interpolación lineal
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                df_imp.loc[idx, col] = serie_interp.loc[idx]
            elif L < MEDIUM_GAP_MAX:
                # Imputación estacional por hora
                if col in hourly_stats_dict:
                    hourly_vals = idx.hour.map(hourly_stats_dict[col])
                    df_imp.loc[idx, col] = hourly_vals.values
                # Si no hay estadístico (columna no presente), se deja NaN (lo tomará KNN)
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols, usando extra_cols como predictores adicionales.
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if extra_cols:
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    imputed_array = imputer.fit_transform(sub)
    df_imp = df.copy()
    df_imp[cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

# ============================================================================
# PROCESAMIENTO POR TRANSECTO
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    # Buscar todos los archivos _outliers.csv
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Diccionario para agrupar por transecto: key = nombre del transecto, value = lista de DataFrames
    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        # Obtener el nombre del transecto de la columna 'Transecto' (debe ser constante)
        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]  # tomamos el primero (debería ser único)
        # Limpiar nombre para usarlo como nombre de archivo (reemplazar espacios por _)
        transect_clean = transect.replace(" ", "_")

        # Añadir columna con el nombre de la estación (si no existe, la creamos)
        if 'Estacion' not in df.columns:
            # Intentar extraer del nombre del archivo (parte después de T? por ejemplo)
            # Pero mejor usar el nombre de archivo como identificador
            df['Estacion'] = station_name

        # Guardar en el diccionario
        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append(df)

    # Procesar cada transecto
    for transect_clean, df_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")
        # Concatenar todas las estaciones del transecto
        df_concat = pd.concat(df_list, axis=0, sort=False)
        # Ordenar por datetime (aunque vienen de diferentes estaciones, el índice es datetime)
        df_concat = df_concat.sort_index()

        # 1. Calcular estadísticos horarios globales del transecto para cada columna
        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_concat.columns:
                continue
            s = df_concat[col].dropna()
            if SEASONAL_STAT == 'mean':
                hourly = s.groupby(s.index.hour).mean()
            else:
                hourly = s.groupby(s.index.hour).median()
            hourly_stats[col] = hourly

        # 2. Aplicar imputación (interpolación y estacional)
        df_imp = impute_dataframe(df_concat, hourly_stats)

        # 3. Preparar para KNN: convertir Estacion a código numérico
        #    Primero aseguramos que la columna Estacion existe
        if 'Estacion' not in df_imp.columns:
            print(f"  Error: no hay columna 'Estacion' en el transecto {transect_clean}")
            continue
        df_imp['station_code'] = pd.factorize(df_imp['Estacion'])[0]
        extra_cols = ['station_code']

        # 4. Aplicar KNN a los NaN restantes (gaps largos)
        df_imp = apply_knn_imputation(df_imp, NUM_COLS, extra_cols=extra_cols)

        # 5. Eliminar la columna auxiliar station_code
        df_imp.drop(columns=['station_code'], inplace=True)

        # 6. Renombrar O3_for_impute a O3
        if 'O3_for_impute' in df_imp.columns:
            df_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)

        # 7. Guardar el DataFrame del transecto completo
        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_imp.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_imp.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (todas las estaciones juntas)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)
        df['station_id'] = station_name
        dfs.append(df)

    df_all = pd.concat(dfs, axis=0)
    # Ordenar por datetime (importante para la imputación)
    df_all = df_all.sort_index()

    # 1. Calcular estadísticos horarios globales
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        if SEASONAL_STAT == 'mean':
            hourly = s.groupby(s.index.hour).mean()
        else:
            hourly = s.groupby(s.index.hour).median()
        global_hourly_stats[col] = hourly

    # 2. Imputar por estación (aplicando la función impute_dataframe a cada una por separado,
    #    pero usando estadísticos globales)
    #    Nota: aunque impute_dataframe trabaja sobre todo el DataFrame, podemos aplicarlo
    #    directamente a df_all, pero la interpolación lineal se hará sobre toda la serie temporal
    #    mezclando estaciones. Eso no es correcto porque la interpolación debe ser intra-estación.
    #    Por tanto, debemos procesar cada estación por separado.
    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        # Aplicar imputación (interpolación + estacional) usando estadísticos globales
        df_station = impute_dataframe(df_station, global_hourly_stats)
        df_all_imp.loc[mask] = df_station

    # 3. KNN global con station_code
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    df_all_imp.drop(columns=['station_code'], inplace=True)

    # 4. Renombrar O3_for_impute a O3
    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)

    # 5. Separar por estación y guardar
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        df_station.drop(columns=['station_id'], inplace=True)
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")